# Natural-Language Querying of the Graph

Different from `02_graph_augmented_qa.ipynb`: there, *we* wrote the Cypher and the LLM answered in English using the results. Here, the full pipeline is automated end to end:

**question → LLM writes Cypher → validated as read-only → run against Neo4j → results fed back to the LLM → answer in plain English**

Every step of that pipeline (Cypher generation, the safety check, execution against Neo4j, and the final answer) is defined directly in this notebook, not imported from a module — so you can show the class exactly what's happening at each stage. `nl_query.py` only holds pure/testable helpers (schema text, prompt building, Cypher extraction, the read-only check, result formatting); `graphrag.py` only holds the Azure OpenAI client and answer-prompt plumbing.

Safety: every generated query is checked against a denylist (`CREATE`, `MERGE`, `DELETE`, `SET`, `CALL`, ...) before it touches the database. If the model ever generates a write, this refuses to run it rather than executing it.

Needs the same `.env` as the other notebooks (`NEO4J_*` + `AZURE_OPENAI_API_KEY`/`AZURE_OPENAI_ENDPOINT`/`AZURE_OPENAI_API_VERSION`/`AZURE_OPENAI_DEPLOYMENT`).

## 1. Connect to Neo4j and Azure OpenAI

In [10]:
from graph_ops import get_driver
from graphrag import get_llm_client

driver = get_driver()
llm = get_llm_client()
print("Connected to Neo4j Aura and Azure OpenAI.")

Connected to Neo4j Aura and Azure OpenAI.


## 2. The schema the model is given

This is the only "knowledge" of the graph the model gets — no examples, no few-shot Cypher. It has to generalize from the labels and relationship types alone.

In [2]:
from nl_query import SCHEMA_DESCRIPTION

print(SCHEMA_DESCRIPTION)

Node labels (every node has a `name` property):
  Person, Company, Department, Project, Location, Client

Relationship types (all directed as shown):
  (Person)-[:IS_HEAD_OF]->(Department)
  (Person)-[:WORKS_IN]->(Department)
  (Department)-[:PART_OF]->(Company)
  (Company)-[:HEADQUARTERED_IN]->(Location)
  (Department)-[:BASED_IN]->(Location)
  (Person)-[:LEADS]->(Project)
  (Person)-[:WORKS_ON]->(Project)
  (Company)-[:PARTNERS_WITH]->(Client)
  (Project)-[:FOR_CLIENT]->(Client)


## 3. The pipeline: question → Cypher → results → answer

Four functions, each doing one visible step:
- `run_query` — execute a Cypher string against Neo4j, return a DataFrame (same helper `01_pdf_to_knowledge_graph.ipynb` uses).
- `generate_cypher` — ask the LLM for Cypher, given the schema from Section 2.
- `run_nl_query` — generate, check it's read-only, then execute.
- `answer_nl_query` — run the above, then hand the raw results back to the LLM as grounding and ask it to answer the original question in plain English.

In [3]:
import pandas as pd
from IPython.display import display

from nl_query import build_nl_to_cypher_prompt, extract_cypher, is_read_only, format_query_results
from graphrag import get_deployment_name, ask_llm


def run_query(driver, cypher, **params):
    """Run a Cypher query and return the results as a DataFrame."""
    with driver.session() as session:
        result = session.run(cypher, **params)
        return pd.DataFrame([record.data() for record in result])


def generate_cypher(client, question):
    """Ask the LLM to translate the question into Cypher."""
    response = client.chat.completions.create(
        model=get_deployment_name(),
        max_completion_tokens=300,
        messages=[{"role": "user", "content": build_nl_to_cypher_prompt(question)}],
    )
    return extract_cypher(response.choices[0].message.content)


def run_nl_query(driver, client, question):
    """Translate `question` to Cypher, refuse anything but a read, then run it."""
    cypher = generate_cypher(client, question)
    if not is_read_only(cypher):
        raise ValueError(f"Refusing to run non-read-only generated Cypher:\n{cypher}")
    return cypher, run_query(driver, cypher)


def answer_nl_query(driver, client, question):
    """Full pipeline: NL question -> generated Cypher -> live results -> NL answer."""
    cypher, results = run_nl_query(driver, client, question)
    context = format_query_results(results)
    answer = ask_llm(client, question, context=context)
    return cypher, results, answer

In [4]:
cypher, results, answer = answer_nl_query(driver, llm, "Which people work in the Data Engineering department?")
print(cypher)
display(results)
print(answer)

MATCH (p:Person)-[:WORKS_IN]->(d:Department)
WHERE d.name = "Data Engineering"
RETURN p.name AS person_name, d.name AS department_name


,person_name,department_name
0,Liam Foster,Data Engineering


Liam Foster works in the Data Engineering department.


In [5]:
cypher, results, answer = answer_nl_query(driver, llm, "What client is the Aurora project for?")
print(cypher)
display(results)
print(answer)

MATCH (p:Project {name: "Aurora"})-[:FOR_CLIENT]->(c:Client)
RETURN p.name AS project_name, c.name AS client_name


,project_name,client_name
0,Aurora,Fortis Bank


The Aurora project is for **Fortis Bank**.


## 4. A harder one: multi-hop, no exact relationship in mind

Liam Foster and Ethan Wallace never appear in the same sentence in the source text. The model has to reach for something like `shortestPath` on its own — nothing in the prompt suggests that pattern.

In [6]:
cypher, results, answer = answer_nl_query(driver, llm, "How, if at all, are Liam Foster and Ethan Wallace connected?")
print(cypher)
display(results)
print(answer)

MATCH p = shortestPath((liam:Person {name: "Liam Foster"})-[*..6]-(ethan:Person {name: "Ethan Wallace"}))
RETURN [n IN nodes(p) | n.name] AS path


,path
0,"[Liam Foster, Data Engineering, Nexora Systems..."


Liam Foster and Ethan Wallace are connected through the path: Liam Foster → Data Engineering → Nexora Systems → Research → Ethan Wallace.


## 5. The safety guard, deliberately triggered

Ask a question shaped like a write. Two things can happen: the model might comply with "read-only only" and just generate a read query instead (in which case you'll see that query and its results — no delete happened, but the guard was never actually tested), or it might generate the delete anyway (in which case `is_read_only` catches it and refuses). Either way, the cell below now prints what actually happened instead of going silent.

To guarantee the class sees the refusal itself fire at least once, the second cell checks `is_read_only` directly against a hand-written destructive query, bypassing the LLM.

In [ ]:
cypher, results = None, None
try:
    cypher, results = run_nl_query(driver, llm, "Delete the Beacon project from the graph.")
except ValueError as e:
    print(f"Refused: {e}")

if cypher is not None:
    print("The model didn't attempt a write for this question — it generated a read-only query instead:")
    print(cypher)
    display(results)

In [8]:
driver.close()